IS 362 – Week 5 Assignment (Tidying up Your Data)

Name: Rezoan

Commentary:
In this notebook I load the nycflights13 CSV files and practice Week 5 skills:
missing values, sorting, filtering, groupby, and reshaping data (wide ↔ long).
Then I answer the 3 questions.

In [1]:
import os

# show files in the current notebook folder
os.listdir()

['.ipynb_checkpoints',
 'airlines.csv',
 'airports.csv',
 'flights.csv',
 'IS362_Week5_Assignment.ipynb',
 'planes.csv',
 'weather.csv']

Load Data (CSV files)

Why:
I load the nycflights13 CSV tables into pandas DataFrames so I can clean, filter,
summarize, and reshape the data.

In [2]:
import pandas as pd
import numpy as np

# load csv files from the same folder as this notebook
airports  = pd.read_csv("airports.csv")
airlines  = pd.read_csv("airlines.csv")
planes    = pd.read_csv("planes.csv")
weather   = pd.read_csv("weather.csv")
flights   = pd.read_csv("flights.csv", low_memory=False)

# quick check shapes
print("airports:", airports.shape)
print("airlines:", airlines.shape)
print("planes:", planes.shape)
print("weather:", weather.shape)
print("flights:", flights.shape)

flights.head()

airports: (1458, 8)
airlines: (16, 2)
planes: (3322, 9)
weather: (26115, 15)
flights: (336776, 20)


,rownames,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,1,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01T10:00:00Z
1,2,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01T10:00:00Z
2,3,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01T10:00:00Z
3,4,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01T10:00:00Z
4,5,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01T11:00:00Z


Missing Values (NaN)

Why:
I check missing values to see which columns have NaN and how much is missing.
This helps decide what to keep or clean later.

In [3]:
# missing values count (top 10)
flights.isna().sum().sort_values(ascending=False).head(10)

air_time     9430
arr_delay    9430
arr_time     8713
dep_time     8255
dep_delay    8255
tailnum      2512
rownames        0
minute          0
hour            0
distance        0
dtype: int64

In [4]:
# percent missing (top 10)
(flights.isna().mean() * 100).sort_values(ascending=False).head(10)

air_time     2.800081
arr_delay    2.800081
arr_time     2.587180
dep_time     2.451184
dep_delay    2.451184
tailnum      0.745896
rownames     0.000000
minute       0.000000
hour         0.000000
distance     0.000000
dtype: float64

Sorting

Why:
I sort the data to quickly find extreme values (example: biggest arrival delays)
and to inspect patterns.

In [5]:
flights.sort_values(by="arr_delay", ascending=False)[
    ["year","month","day","carrier","flight","origin","dest","dep_delay","arr_delay"]
].head(10)

,year,month,day,carrier,flight,origin,dest,dep_delay,arr_delay
7072,2013,1,9,HA,51,JFK,HNL,1301.0,1272.0
235778,2013,6,15,MQ,3535,JFK,CMH,1137.0,1127.0
8239,2013,1,10,MQ,3695,EWR,ORD,1126.0,1109.0
327043,2013,9,20,AA,177,JFK,SFO,1014.0,1007.0
270376,2013,7,22,MQ,3075,JFK,CVG,1005.0,989.0
173992,2013,4,10,DL,2391,JFK,TPA,960.0,931.0
151974,2013,3,17,DL,2119,LGA,MSP,911.0,915.0
270987,2013,7,22,DL,2047,LGA,ATL,898.0,895.0
87238,2013,12,5,AA,172,EWR,MIA,896.0,878.0
195711,2013,5,3,MQ,3744,EWR,ORD,878.0,875.0


In [6]:
flights.sort_values(by="sched_dep_time", ascending=True)[
    ["year","month","day","carrier","flight","origin","dest","sched_dep_time"]
].head(10)

,year,month,day,carrier,flight,origin,dest,sched_dep_time
275945,2013,7,27,US,1632,EWR,LGA,106
40020,2013,10,15,US,1843,EWR,CLT,500
256653,2013,7,8,US,1431,EWR,CLT,500
137970,2013,3,3,US,1113,EWR,CLT,500
317276,2013,9,10,US,1877,EWR,CLT,500
292344,2013,8,14,US,1993,EWR,CLT,500
257650,2013,7,9,US,1431,EWR,CLT,500
192451,2013,4,30,US,1219,EWR,CLT,500
136250,2013,3,1,US,1117,EWR,CLT,500
137206,2013,3,2,US,1117,EWR,CLT,500


Filtering

Why:
I filter rows to focus on specific cases (JFK flights, big delays, OR condition).
This is useful for answering specific questions.

In [7]:
jfk_flights = flights[flights["origin"] == "JFK"]
jfk_flights[["year","month","day","carrier","flight","origin","dest","dep_delay","arr_delay"]].head(10)

,year,month,day,carrier,flight,origin,dest,dep_delay,arr_delay
2,2013,1,1,AA,1141,JFK,MIA,2.0,33.0
3,2013,1,1,B6,725,JFK,BQN,-1.0,-18.0
8,2013,1,1,B6,79,JFK,MCO,-3.0,-8.0
10,2013,1,1,B6,49,JFK,PBI,-2.0,-2.0
11,2013,1,1,B6,71,JFK,TPA,-2.0,-3.0
12,2013,1,1,UA,194,JFK,LAX,-2.0,7.0
15,2013,1,1,B6,1806,JFK,BOS,0.0,-4.0
23,2013,1,1,DL,1743,JFK,ATL,-4.0,-8.0
26,2013,1,1,UA,303,JFK,SFO,11.0,14.0
27,2013,1,1,B6,135,JFK,RSW,3.0,4.0


In [8]:
jfk_big_delay = flights[(flights["origin"] == "JFK") & (flights["arr_delay"] > 60)]
jfk_big_delay[["year","month","day","carrier","flight","origin","dest","dep_delay","arr_delay"]].head(10)

,year,month,day,carrier,flight,origin,dest,dep_delay,arr_delay
151,2013,1,1,MQ,3944,JFK,BWI,853.0,851.0
373,2013,1,1,B6,673,JFK,LAX,77.0,78.0
411,2013,1,1,B6,355,JFK,BUR,59.0,83.0
491,2013,1,1,B6,705,JFK,SJU,122.0,115.0
512,2013,1,1,EV,5712,JFK,IAD,119.0,123.0
542,2013,1,1,B6,63,JFK,TPA,88.0,80.0
593,2013,1,1,B6,703,JFK,SJU,91.0,61.0
617,2013,1,1,9E,3651,JFK,RDU,88.0,66.0
680,2013,1,1,AA,177,JFK,SFO,63.0,78.0
689,2013,1,1,AA,181,JFK,LAX,131.0,127.0


In [9]:
jfk_or_lga = flights[(flights["origin"] == "JFK") | (flights["origin"] == "LGA")]
jfk_or_lga[["origin","dest","carrier","flight"]].head(10)

,origin,dest,carrier,flight
1,LGA,IAH,UA,1714
2,JFK,MIA,AA,1141
3,JFK,BQN,B6,725
4,LGA,ATL,DL,461
7,LGA,IAD,EV,5708
8,JFK,MCO,B6,79
9,LGA,ORD,AA,301
10,JFK,PBI,B6,49
11,JFK,TPA,B6,71
12,JFK,LAX,UA,194


Groupby

Why:
I use groupby to summarize data (average delay by carrier and by origin).
This is a basic analysis step for tidy data.

In [10]:
flights.groupby("carrier")["arr_delay"].mean().sort_values(ascending=False).head(10)

carrier
F9    21.920705
FL    20.115906
EV    15.796431
YV    15.556985
OO    11.931034
MQ    10.774733
WN     9.649120
B6     9.457973
9E     7.379669
UA     3.558011
Name: arr_delay, dtype: float64

In [11]:
flights.groupby("origin")["arr_delay"].mean().sort_values(ascending=False)

origin
EWR    9.107055
LGA    5.783488
JFK    5.551481
Name: arr_delay, dtype: float64

In [12]:
carrier_delay = flights.groupby("carrier", as_index=False)["arr_delay"].mean()
carrier_delay = carrier_delay.merge(airlines, on="carrier", how="left")
carrier_delay.sort_values("arr_delay", ascending=False).head(10)

,carrier,arr_delay,name
6,F9,21.920705,Frontier Airlines Inc.
7,FL,20.115906,AirTran Airways Corporation
5,EV,15.796431,ExpressJet Airlines Inc.
15,YV,15.556985,Mesa Airlines Inc.
10,OO,11.931034,SkyWest Airlines Inc.
9,MQ,10.774733,Envoy Air
14,WN,9.649120,Southwest Airlines Co.
3,B6,9.457973,JetBlue Airways
0,9E,7.379669,Endeavor Air Inc.
11,UA,3.558011,United Air Lines Inc.


Wide ↔ Long (Tidying)

Why:
Wide format has months as columns. Long format (tidy) has month as a single column.
Long format is easier to group, filter, and analyze.

In [13]:
origin_month = flights.groupby(["origin", "month"], as_index=False)["arr_delay"].mean()
origin_month.head()

,origin,month,arr_delay
0,EWR,1,12.816556
1,EWR,2,8.775160
2,EWR,3,10.600799
3,EWR,4,14.143388
4,EWR,5,5.381928


In [14]:
wide_delay = origin_month.pivot(index="origin", columns="month", values="arr_delay")
wide_delay

month,1,2,3,4,5,6,7,8,9,10,11,12
origin,,,,,,,,,,,,
EWR,12.816556,8.775160,10.600799,14.143388,5.381928,16.863599,15.460201,6.712342,-4.729972,2.604737,0.672498,19.639745
JFK,1.368398,4.391033,2.580815,7.011539,2.122977,17.596929,20.190222,5.910841,-4.463018,-3.585972,-0.872874,12.677575
LGA,3.382402,3.147389,3.738498,12.038582,2.796376,14.769278,14.181570,5.407801,-2.825395,0.186423,1.551187,11.956372


In [15]:
wide_delay_reset = wide_delay.reset_index()

tidy_delay = wide_delay_reset.melt(
    id_vars="origin",
    var_name="month",
    value_name="avg_arr_delay"
)

tidy_delay.head(10)

,origin,month,avg_arr_delay
0,EWR,1,12.816556
1,JFK,1,1.368398
2,LGA,1,3.382402
3,EWR,2,8.775160
4,JFK,2,4.391033
5,LGA,2,3.147389
6,EWR,3,10.600799
7,JFK,3,2.580815
8,LGA,3,3.738498
9,EWR,4,14.143388


In [16]:
tidy_delay.sort_values("avg_arr_delay", ascending=False).head(10)

,origin,month,avg_arr_delay
19,JFK,7,20.190222
33,EWR,12,19.639745
16,JFK,6,17.596929
15,EWR,6,16.863599
18,EWR,7,15.460201
17,LGA,6,14.769278
20,LGA,7,14.181570
9,EWR,4,14.143388
0,EWR,1,12.816556
34,JFK,12,12.677575


Question 1:
What is the northernmost airport in the United States?

Why:
I filter to US timezones and realistic US longitude, then sort by latitude.

In [17]:
# filter to US timezone airports
us_airports = airports[airports["tzone"].str.startswith("America/", na=False)]

# for northernmost: keep western hemisphere (lon < 0) to avoid weird results
us_airports_west = us_airports[us_airports["lon"] < 0]

north5 = us_airports_west.sort_values("lat", ascending=False)[["faa","name","lat","lon","tzone"]].head(5)
north5

,faa,name,lat,lon,tzone
230,BRW,Wiley Post Will Rogers Mem,71.285446,-156.766003,America/Anchorage
110,AIN,Wainwright Airport,70.638056,-159.994722,America/Anchorage
708,K03,Wainwright As,70.613378,-159.860350,America/Anchorage
152,ATK,Atqasuk Edward Burnell Sr Memorial Airport,70.467300,-157.436000,America/Anchorage
1363,UUK,Ugnu-Kuparuk Airport,70.330833,-149.597500,America/Anchorage


Answer Q1:
The northernmost airport in the United States is BRW — Wiley Post–Will Rogers Memorial Airport.
I ignored EEN because its coordinates/timezone look incorrect.

Question 2:
What is the easternmost airport in the United States?

Why:
I filter to US timezones and sort by longitude (largest lon = farthest east).

In [18]:
east10 = us_airports.sort_values("lon", ascending=False)[["faa","name","lat","lon","tzone"]].head(10)
east10

,faa,name,lat,lon,tzone
1290,SYA,Eareckson As,52.712275,174.113620,America/Anchorage
444,EPM,Eastport Municipal Airport,44.910111,-67.012694,America/New_York
624,HUL,Houlton Intl,46.123083,-67.792056,America/New_York
259,CAR,Caribou Muni,46.871500,-68.017917,America/New_York
1101,PQI,Northern Maine Rgnl At Presque Isle,46.688958,-68.044797,America/New_York
1398,WFK,Northern Aroostook Regional Airport,47.285556,-68.312778,America/New_York
192,BHB,Hancock County - Bar Harbor,44.449769,-68.361565,America/New_York
856,ME5,Banks Airport,44.165389,-68.428167,America/New_York
894,MLT,Millinocket Muni,45.647836,-68.685561,America/New_York
191,BGR,Bangor Intl,44.807444,-68.828139,America/New_York


Answer Q2:
The easternmost airport in the United States is SYA — Eareckson Air Station (lon 174.113620, Alaska).
This happens because some Aleutian airports cross into positive longitudes.

Question 3:
On February 12th, 2013, which New York area airport had the windiest weather?

Why:
I compare wind_speed for EWR, JFK, and LGA on that date.
I remove an extreme outlier wind_speed value (1048) by setting wind_speed > 200 to NaN.
Then I compare the max wind_speed by airport.

In [19]:
nyc_weather = weather[
    (weather["year"] == 2013) &
    (weather["month"] == 2) &
    (weather["day"] == 12) &
    (weather["origin"].isin(["EWR", "JFK", "LGA"]))
]

nyc_weather[["origin","hour","wind_speed","wind_gust"]].sort_values("wind_speed", ascending=False).head(10)

,origin,hour,wind_speed,wind_gust
1009,EWR,3,1048.36058,NaN
18417,LGA,2,23.01560,31.07106
1018,EWR,12,21.86482,31.07106
18428,LGA,13,21.86482,25.31716
18429,LGA,14,20.71404,25.31716
1008,EWR,2,20.71404,25.31716
9717,JFK,8,20.71404,27.61872
9716,JFK,7,19.56326,NaN
18427,LGA,12,19.56326,26.46794
18416,LGA,1,19.56326,28.76950


In [20]:
nyc_weather_clean = nyc_weather.copy()

# outlier fix: wind_speed above 200 is not realistic
nyc_weather_clean.loc[nyc_weather_clean["wind_speed"] > 200, "wind_speed"] = np.nan

nyc_weather_clean.groupby("origin")["wind_speed"].max().sort_values(ascending=False)

origin
LGA    23.01560
EWR    21.86482
JFK    20.71404
Name: wind_speed, dtype: float64

Answer Q3:
On Feb 12, 2013, the windiest NYC-area airport was LGA (LaGuardia) with max wind_speed 23.01560.
I removed one extreme outlier wind_speed value (1048) by setting any wind_speed > 200 to NaN,
then I compared the max wind speed by airport.

Conclusion:
The notebook runs clean in Jupyter. The code and commentary explain each step and answers match the prompts.